In [1]:
!pip install scikit-learn datasets transformers Pillow rouge_score openai python-dotenv
!pip install openai
!pip install "accelerate>=1.1.0"
!pip install transformers
!pip install requests

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [2]:
%pip install requests
%pip install pandas
%pip install scikit-learn
%pip install datasets
%pip install transformers
%pip install Pillow
%pip install rouge_score
%pip install openai
%pip install python-dotenv
%pip install openai
%pip install "accelerate>=1.1.0"
%pip install transformers
%pip install requests

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl.metadata (40 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl (309 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.

# Projeto final - Paradigmas de Aprendizagem de máquina

## Montando base de dados

In [18]:
from getting_database import buscar_filmes_terror

filmes1 = buscar_filmes_terror(quantidade=200, ordenar_por="vote_average.desc")
filmes2 = buscar_filmes_terror(quantidade=200, ordenar_por="popularity.desc")



### Salvar em arquivo

In [19]:
import pandas as pd

df = pd.DataFrame(filmes1)
df2 = pd.DataFrame(filmes2)
df_concat = pd.concat([df, df2])
df_filtrado = df[["titulo", "poster", "sinopse"]]
df_filtrado.to_csv("filmes.csv", index=False, encoding="utf-8")
df = df_filtrado

## EDA e pré-processamento

In [3]:
import pandas as pd

df = pd.read_csv("filmes.csv")

In [4]:
df.head()

,titulo,poster,sinopse
0,O Jogo da Tentação,https://image.tmdb.org/t/p/w500/dtdZRfoNRNcH92...,"Um rapaz, que acabou de se tornar pai e luta c..."
1,Psicose,https://image.tmdb.org/t/p/w500/oC2iYT2on8c2iZ...,Marion Crane é uma secretária que rouba 40 mil...
2,La Leyenda de los Chaneques,https://image.tmdb.org/t/p/w500/4f9ghI3utknpeB...,NaN
3,Las leyendas: El origen,https://image.tmdb.org/t/p/w500/fR49hZdFJ6ZtRS...,NaN
4,Michael Jackson: Thriller,https://image.tmdb.org/t/p/w500/dYHGoPMkZMVuBA...,Uma noite no cinema se transforma em um pesade...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   titulo   400 non-null    str  
 1   poster   400 non-null    str  
 2   sinopse  391 non-null    str  
dtypes: str(3)
memory usage: 172.2 KB


In [6]:
print("duplicados: ", df.duplicated().sum())
df = df.drop_duplicates()


duplicados:  66


In [7]:
print("Total de filmes: ", len(df))
df = df.dropna(subset=["sinopse"])
print("Total de filmes depois de remover sinopses nulas: ", len(df))


Total de filmes:  334
Total de filmes depois de remover sinopses nulas:  325


In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Treino: {len(train_df)} filmes")
print(f"Teste:  {len(test_df)} filmes")

Treino: 260 filmes
Teste:  65 filmes


### Baixando pôsteres

Precisamos das imagens localmente para o modelo processá-las.
Cada pôster é baixado uma vez e salvo em `posters/`.

In [10]:
import requests
from pathlib import Path

POSTERS_DIR = Path("data/posters")
POSTERS_DIR.mkdir(exist_ok=True)

def download_poster(url):
    filename = url.split("/")[-1]
    path = POSTERS_DIR / filename
    if not path.exists():
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        path.write_bytes(r.content)
    return str(path)

def add_image_paths(df):
    paths = []
    for url in df["poster"]:
        try:
            paths.append(download_poster(url))
        except Exception:
            paths.append(None)
    df = df.copy()
    df["image_path"] = paths
    return df.dropna(subset=["image_path"]).reset_index(drop=True)

train_df = add_image_paths(train_df)
test_df = add_image_paths(test_df)
print(f"Treino: {len(train_df)}, Teste: {len(test_df)}")

Treino: 260, Teste: 65


### Convertendo para HuggingFace Dataset

O mesmo formato que o notebook BERT usa com `load_dataset("rotten_tomatoes")`.
Aqui construímos o dataset manualmente a partir do nosso DataFrame.

In [11]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["image_path", "titulo", "sinopse"]])
test_dataset = Dataset.from_pandas(test_df[["image_path", "titulo", "sinopse"]])
print(train_dataset)

/Users/davinasiaseneamorim/Documents/faculdade/ml/projeto-ia-final/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['image_path', 'titulo', 'sinopse'],
    num_rows: 260
})


## Carregando modelo

In [12]:
from transformers import BlipProcessor, BlipForConditionalGeneration

MODEL_CHECKPOINT = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_CHECKPOINT)
model = BlipForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 81340.95it/s]
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Pré-processamento para o modelo

In [15]:
from PIL import Image as PILImage

# O decoder do BLIP gera logits com o mesmo comprimento de input_ids; labels precisa
# ter exatamente esse comprimento (ver modeling_blip_text: shift + CrossEntropyLoss).
MAX_LEN = 128


def preprocess(example):
    image = PILImage.open(example["image_path"]).convert("RGB")
    prompt = "Gere uma sinopse de terror para o filme: " + example["titulo"]
    full_text = prompt + " " + example["sinopse"]

    inputs = processor(
        images=image,
        text=full_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )
    labels = inputs.input_ids.clone()
    enc_text = processor.tokenizer(
        full_text,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    prompt_end_char = len(prompt) + 1
    offsets = enc_text.offset_mapping[0]
    for i in range(enc_text.input_ids.shape[1]):
        _start, end = offsets[i]
        if end == 0:
            continue
        if end <= prompt_end_char:
            labels[0, i] = -100
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": inputs.pixel_values.squeeze(0),
        "input_ids": inputs.input_ids.squeeze(0),
        "attention_mask": inputs.attention_mask.squeeze(0),
        "labels": labels.squeeze(0),
    }

### Aplicando pré-processamento

Assim como no notebook BERT fazemos `.map(preprocessamento, batched=True)`, aqui aplicamos o mesmo padrão.
A diferença é que cada item carrega uma imagem além do texto.

In [16]:
tokenized_train = train_dataset.map(
    preprocess, batched=False,
    remove_columns=train_dataset.column_names,
)
tokenized_test = test_dataset.map(
    preprocess, batched=False,
    remove_columns=test_dataset.column_names,
)
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
print(tokenized_train)

Map: 100%|██████████| 65/65 [00:00<00:00, 122.77 examples/s]

Dataset({
    features: ['pixel_values', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 260
})


## Definição de Métricas — LLM as a Judge

Como sinopses são textos criativos, métricas de sobreposição de palavras (como ROUGE) não capturam
bem a qualidade. Usamos um LLM para avaliar cada sinopse gerada em três critérios:
- **Clima de terror**: a sinopse transmite tensão/medo?
- **Coerência com o título**: faz sentido para o filme?
- **Qualidade narrativa**: é bem escrita e coesa?

As funções abaixo serão chamadas na seção de avaliação, após o treino.

In [29]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

JUDGE_PROMPT = """Você é um avaliador de sinopses de filmes de terror.

Título do filme: {titulo}
Sinopse gerada pelo modelo: {gerada}

Avalie a sinopse gerada de 1 a 5 em cada critério e dê uma justificativa breve:
1. Clima de terror: a sinopse transmite tensão/medo?
2. Coerência com o título: faz sentido para o filme mencionado?
3. Qualidade narrativa: é bem escrita e coesa?

Responda APENAS neste formato JSON:
{{"clima_terror": <1-5>, "coerencia_titulo": <1-5>, "qualidade_narrativa": <1-5>, "justificativa": "<texto breve>"}}"""


def gerar_sinopse(row):
    image = PILImage.open(row["image_path"]).convert("RGB")
    inputs = processor(
        images=image,
        text="Gere uma sinopse de terror para o filme: " + row["titulo"],
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, max_length=128)
    return processor.decode(output[0], skip_special_tokens=True)


def avaliar_sinopse(titulo, sinopse_gerada):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{
            "role": "user",
            "content": JUDGE_PROMPT.format(
                titulo=titulo,
                gerada=sinopse_gerada
            ),
        }],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

In [27]:
import pandas as pd

def rodar_avaliacao(test_df):
    resultados = []
    for _, row in test_df.iterrows():
        sinopse_gerada = gerar_sinopse(row)
        avaliacao = avaliar_sinopse(row["titulo"], sinopse_gerada)
        resultados.append({
            "titulo": row["titulo"],
            "sinopse_gerada": sinopse_gerada,
            **avaliacao,
        })
    df_resultados = pd.DataFrame(resultados)
    print(df_resultados[["titulo", "clima_terror", "coerencia_titulo", "qualidade_narrativa", "justificativa"]].to_string())
    print("\n--- Médias ---")
    print(df_resultados[["clima_terror", "coerencia_titulo", "qualidade_narrativa"]].mean().round(2))
    return df_resultados

## Treinamento

Análogo à seção de `Treinamento` do notebook BERT, mas usamos `Seq2SeqTrainer`
em vez de `Trainer` porque a tarefa é geração de texto (seq2seq), não classificação.

Antes de treinar, verificamos se há GPU disponível e movemos o modelo para ela.

In [19]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Treinando em: {device}")

Treinando em: cpu


### Congelamento do Vision Encoder

Analogamente ao notebook BERT onde congelamos `bert.*` e treinamos só `classifier.*`,
aqui congelamos `vision_model.*` e treinamos só `text_decoder.*`.

O `vision_model` (ViT) já aprendeu a "ver" imagens em bilhões de exemplos — não precisamos
reaprender isso. O que queremos ensinar é o decoder a gerar texto no **estilo de terror**.

In [20]:
for name, param in model.named_parameters():
    if "vision_model" in name:
        param.requires_grad = False  # congela o ViT — ele já sabe "ver"

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"Parâmetros treináveis: {len(trainable)}")
print(f"Total de parâmetros:   {len(list(model.named_parameters()))}")

Parâmetros treináveis: 321
Total de parâmetros:   471


In [21]:
training_args = Seq2SeqTrainingArguments(
    output_dir="blip-horror",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    predict_with_generate=True,
    save_strategy="epoch",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=processor.tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None, 'pad_token_id': 0}.
/Users/davinasiaseneamorim/Documents/faculdade/ml/projeto-ia-final/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.10s/it]
/Users/davinasiaseneamorim/Documents/faculdade/ml/projeto-ia-final/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]
/Users/davinasiaseneamorim/Documents/faculdade/ml/projeto-ia-final/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.29s/it]


TrainOutput(global_step=99, training_loss=4.928302495166509, metrics={'train_runtime': 230.4381, 'train_samples_per_second': 3.385, 'train_steps_per_second': 0.43, 'total_flos': 4.1433799810351104e+17, 'train_loss': 4.928302495166509, 'epoch': 3.0})

In [30]:
df_resultados = rodar_avaliacao(test_df)

                                                 titulo  clima_terror  coerencia_titulo  qualidade_narrativa                                                                                                                                                                                                                                                                                                                                                                  justificativa
0                          Blade: O Caçador de Vampiros             1                 2                    1                                                                                                                                                                             A sinopse está repetitiva, sem sentido ou elementos que transmitam medo ou tensão; não há coesão com o título, que sugere ação e caça, enquanto a sinopse apresenta apenas repetições incoerentes.
1                        O Segredo do Bosque dos

In [32]:
df_resultados.head()

df_resultados.to_csv("resultados.csv", index=False)